In [73]:
from pathlib import Path
import geopandas as gpd
import pandas as pd
import folium

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
BOUNDARIES_DIR = DATA_DIR / "boundaries"
TOC_DIR = BOUNDARIES_DIR / "tocs"
VILLAGES_DIR = BOUNDARIES_DIR / "villages"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"

tocs_gpkg = TOC_DIR / "toc_demographics.gpkg"
villages_gpkg = VILLAGES_DIR / "village_demographics.gpkg"

In [74]:
tocs_merged = gpd.read_file(tocs_gpkg)
villages_merged = gpd.read_file(villages_gpkg)

In [75]:
tocs_merged.columns

Index(['OBJECTID', 'APPLICABIL', 'TOC_ID', 'income_weighted', 'rent_weighted',
       'hh_total', 'median_income_approx', 'median_rent_approx',
       'intersect_acres', 'ASNQE001', 'ASS8E001', 'ASS8E002', 'ASS8E003',
       'ASS9E002', 'ASS9E003', 'ASORE001', 'ASORE003', 'ASORE004', 'ASORE010',
       'ASORE018', 'ASORE019', 'ASORE021', 'ASORE002', 'pct_drive_alone',
       'pct_transit', 'pct_bike', 'pct_walk', 'pct_wfh', 'pct_auto',
       'geometry'],
      dtype='object')

In [76]:
villages_merged.columns

Index(['OBJECTID', 'NAME', 'ASNQE001_w', 'ASS8E002_w', 'ASS8E003_w',
       'intersect_acres', 'pop_per_acre', 'hh_per_acre', 'ASS9E002_w',
       'ASS9E003_w', 'median_income_approx', 'median_rent_approx',
       'pct_drive_alone', 'pct_transit', 'pct_bike', 'pct_walk', 'pct_wfh',
       'pct_auto', 'geometry'],
      dtype='object')

In [77]:
tocs_merged["pct_renter"] = (tocs_merged["ASS9E003"] / tocs_merged["ASS8E001"]) * 100

tocs_merged["pct_owner"] = (tocs_merged["ASS9E002"] / tocs_merged["ASS8E001"]) * 100

tocs_merged["pct_occupied"] = (tocs_merged["ASS8E002"] / tocs_merged["ASS8E001"]) * 100

tocs_merged["pct_vacant"] = (tocs_merged["ASS8E003"] / tocs_merged["ASS8E001"]) * 100

tocs_merged["rent-to-income"] = ((tocs_merged["median_rent_approx"] * 12) / tocs_merged["median_income_approx"]) * 100

In [78]:
# Reproject to WGS84 for web maps
tocs_web     = tocs_merged.to_crs(4326).copy()
villages_web = villages_merged.to_crs(4326).copy()

# --- Nice labels for tooltips ---
def fmt_dollars(v):
    return f"${v:,.0f}" if pd.notnull(v) else "N/A"

def fmt_rate(v):
    return f"{v:,.1f}%" if pd.notnull(v) else "N/A"

def fmt_density(v):
    return f"{v:,.2f}" if pd.notnull(v) else "N/A"

# Villages
villages_web["median_income_label"] = villages_web["median_income_approx"].map(fmt_dollars)
villages_web["median_rent_label"]   = villages_web["median_rent_approx"].map(fmt_dollars)
villages_web["pop_per_acre_label"]  = villages_web["pop_per_acre"].map(fmt_density)
villages_web["hh_per_acre_label"]   = villages_web["hh_per_acre"].map(fmt_density)

# TOCs (same pattern, optional)
tocs_web["median_income_label"] = tocs_web["median_income_approx"].map(fmt_dollars)
tocs_web["median_rent_label"]   = tocs_web["median_rent_approx"].map(fmt_dollars)
tocs_web["pct_transit_label"] = tocs_web["pct_transit"].map(fmt_rate)
tocs_web["pct_walk_label"] = tocs_web["pct_walk"].map(fmt_rate)
tocs_web["pct_wfh_label"] = tocs_web["pct_wfh"].map(fmt_rate)
tocs_web["pct_auto_label"] = tocs_web["pct_auto"].map(fmt_rate)
tocs_web["pct_renter_label"] = tocs_web["pct_renter"].map(fmt_rate)
tocs_web["pct_owner_label"] = tocs_web["pct_owner"].map(fmt_rate)
tocs_web["pct_occupied_label"] = tocs_web["pct_occupied"].map(fmt_rate)
tocs_web["pct_vacant_label"] = tocs_web["pct_vacant"].map(fmt_rate)
tocs_web["rent_to_income_label"] = tocs_web["rent-to-income"].map(fmt_rate)

In [79]:
# Map 1 - median income

# --- Base: villages in grey for context ---
m_income = villages_web.explore(
    style_kwds={
        "fillOpacity": 0.2,
        "weight": 0.3,
        "color": "grey",
    },
    tooltip=["NAME"],
    name="Villages",
)

# --- Overlay: TOCs ---
tocs_web.explore(
    m=m_income,
    column="median_income_approx",
    cmap="Reds",
    legend=True,
    style_kwds={
        "fillOpacity": 0.5,
        "weight": 0.5,
        "color": "black",
    },
    tooltip=[
        "APPLICABIL",
        "median_income_label",
        "median_rent_label",
        "rent_to_income_label",
    ],
    name="TOCs – Median Income",
)

folium.LayerControl(collapsed=False).add_to(m_income)

income_map_path = OUTPUTS_DIR / "tocs_vs_villages_median_income.html"
m_income.save(income_map_path)

print("Saved:", income_map_path)

Saved: c:\Users\arjav\DevSource\toc-performance-dashboard\outputs\tocs_vs_villages_median_income.html


In [80]:
# Map 2 - transit mode share

# --- Base: villages in grey for context ---
m_transit = villages_web.explore(
    style_kwds={
        "fillOpacity": 0.2,
        "weight": 0.3,
        "color": "grey",
    },
    tooltip=["NAME"],
    name="Villages",
)

# --- TOC choropleth by transit mode share ---
tocs_web.explore(
    m=m_transit,
    column="pct_transit",
    cmap="Blues",
    legend=True,
    legend_kwds={"caption": "Transit Mode Share (%)"},
    style_kwds={
        "fillOpacity": 0.7,
        "weight": 0.7,
        "color": "black",
    },
    tooltip=[
        "APPLICABIL",
        "pct_transit_label",
        "pct_auto_label"
    ],
    name="TOCs – Transit Share",
)

folium.LayerControl(collapsed=False).add_to(m_transit)

transit_map_path = OUTPUTS_DIR / "tocs_transit_mode_share.html"
m_transit.save(transit_map_path)

print("Saved:", transit_map_path)

Saved: c:\Users\arjav\DevSource\toc-performance-dashboard\outputs\tocs_transit_mode_share.html


In [81]:
# Map 3 - mode share (WFH)

# --- Base: villages in grey for context ---
m_wfh = villages_web.explore(
    style_kwds={
        "fillOpacity": 0.2,
        "weight": 0.3,
        "color": "grey",
    },
    tooltip=["NAME"],
    name="Villages",
)

# --- TOC choropleth by work from home mode share ---
tocs_web.explore(
    m=m_wfh,
    column="pct_wfh",   # or pct_walk / pct_auto / etc
    cmap="Blues",
    legend=True,
    legend_kwds={"caption": "Work From Home Mode Share (%)"},
    style_kwds={
        "fillOpacity": 0.7,
        "weight": 0.7,
        "color": "black",
    },
    tooltip=[
        "APPLICABIL",
        "pct_wfh_label",
        "pct_auto_label",
    ],
    name="TOCs – Work From Home Share",
)

folium.LayerControl(collapsed=False).add_to(m_wfh)

wfh_map_path = OUTPUTS_DIR / "tocs_wfh_mode_share.html"
m_wfh.save(wfh_map_path)

print("Saved:", wfh_map_path)

Saved: c:\Users\arjav\DevSource\toc-performance-dashboard\outputs\tocs_wfh_mode_share.html


In [82]:
# Map 4 - renter share %

# --- Base: villages in grey for context ---
m_renter = villages_web.explore(
    style_kwds={
        "fillOpacity": 0.2,
        "weight": 0.3,
        "color": "grey",
    },
    tooltip=["NAME"],
    name="Villages",
)

# --- TOC choropleth by renter share ---
tocs_web.explore(
    m=m_renter,
    column="pct_renter",
    cmap="Blues",
    legend=True,
    legend_kwds={"caption": "Renter Share (%)"},
    style_kwds={
        "fillOpacity": 0.7,
        "weight": 0.7,
        "color": "black",
    },
    tooltip=[
        "APPLICABIL",
        "median_income_label",
        "median_rent_label",
        "pct_renter_label",
        "pct_owner_label",
        "pct_occupied_label",
        "pct_vacant_label"
    ],
    name="TOCs – Renter Share",
)

folium.LayerControl(collapsed=False).add_to(m_renter)

renter_map_path = OUTPUTS_DIR / "tocs_renter_share.html"
m_renter.save(renter_map_path)

print("Saved:", renter_map_path)

Saved: c:\Users\arjav\DevSource\toc-performance-dashboard\outputs\tocs_renter_share.html


In [83]:
# Map 5 - vacancy share %

# --- Base: villages in grey for context ---
m_vacant = villages_web.explore(
    style_kwds={
        "fillOpacity": 0.2,
        "weight": 0.3,
        "color": "grey",
    },
    tooltip=["NAME"],
    name="Villages",
)

# --- TOC choropleth by vacancy share ---
tocs_web.explore(
    m=m_vacant,
    column="pct_vacant",
    cmap="Blues",
    legend=True,
    legend_kwds={"caption": "Vacancy Share (%)"},
    style_kwds={
        "fillOpacity": 0.7,
        "weight": 0.7,
        "color": "black",
    },
    tooltip=[
        "APPLICABIL",
        "median_income_label",
        "median_rent_label",
        "pct_renter_label",
        "pct_owner_label",
        "pct_occupied_label",
        "pct_vacant_label"
    ],
    name="TOCs – Vacancy Share",
)

folium.LayerControl(collapsed=False).add_to(m_vacant)

vacancy_map_path = OUTPUTS_DIR / "tocs_vacancy_share.html"
m_vacant.save(vacancy_map_path)

print("Saved:", vacancy_map_path)

Saved: c:\Users\arjav\DevSource\toc-performance-dashboard\outputs\tocs_vacancy_share.html


In [84]:
# Map 6 - median rent

# --- Base: villages in grey for context ---
m_rent = villages_web.explore(
    style_kwds={
        "fillOpacity": 0.2,
        "weight": 0.3,
        "color": "grey",
    },
    tooltip=["NAME"],
    name="Villages",
)

# --- Overlay: TOCs ---
tocs_web.explore(
    m=m_rent,
    column="median_rent_approx",
    cmap="Reds",
    legend=True,
    style_kwds={
        "fillOpacity": 0.5,
        "weight": 0.5,
        "color": "black",
    },
    tooltip=[
        "APPLICABIL",
        "median_rent_label",
        "median_income_label",
        "rent_to_income_label"
    ],
    name="TOCs – Median Rent",
)

folium.LayerControl(collapsed=False).add_to(m_rent)

rent_map_path = OUTPUTS_DIR / "tocs_vs_villages_median_rent.html"
m_rent.save(rent_map_path)

print("Saved:", rent_map_path)

Saved: c:\Users\arjav\DevSource\toc-performance-dashboard\outputs\tocs_vs_villages_median_rent.html


In [85]:
# Map 6 - rent-to-income ratio

# --- Base: villages in grey for context ---
m_rent_to_income = villages_web.explore(
    style_kwds={
        "fillOpacity": 0.2,
        "weight": 0.3,
        "color": "grey",
    },
    tooltip=["NAME"],
    name="Villages",
)

# --- Overlay: TOCs ---
tocs_web.explore(
    m=m_rent_to_income,
    column="rent-to-income",
    cmap="Reds",
    legend=True,
    style_kwds={
        "fillOpacity": 0.5,
        "weight": 0.5,
        "color": "black",
    },
    tooltip=[
        "APPLICABIL",
        "median_rent_label",
        "median_income_label",
        "rent_to_income_label"
    ],
    name="TOCs – Rent-to-Income Ratio",
)

folium.LayerControl(collapsed=False).add_to(m_rent_to_income)

rent_to_income_map_path = OUTPUTS_DIR / "tocs_vs_villages_rent_to_income.html"
m_rent_to_income.save(rent_to_income_map_path)

print("Saved:", rent_to_income_map_path)

Saved: c:\Users\arjav\DevSource\toc-performance-dashboard\outputs\tocs_vs_villages_rent_to_income.html
